In [ ]:
# Cell 0: CRITICAL - Patch numpy BEFORE any other imports
# numpy 2.x removed _center from numpy._core.umath but transformers needs it
import numpy as np

def _center_patch(a, width, fillchar=' '):
    try:
        return a.center(width, fillchar)
    except AttributeError:
        return str(a).center(width, fillchar)

import numpy._core.umath as umath_mod
if not hasattr(umath_mod, '_center'):
    setattr(umath_mod, '_center', _center_patch)
    print('[patch] Added _center to numpy._core.umath', flush=True)

try:
    import numpy._core.defchararray as char_mod
    if not hasattr(char_mod, '_center'):
        setattr(char_mod, '_center', _center_patch)
        print('[patch] Added _center to numpy._core.defchararray', flush=True)
except ImportError:
    pass

# Also patch _blas_supports_fpe for numpy 2.x compatibility
import numpy._core._multiarray_umath as multiarray_mod
if not hasattr(multiarray_mod, '_blas_supports_fpe'):
    setattr(multiarray_mod, '_blas_supports_fpe', lambda x: False)
    print('[patch] Added _blas_supports_fpe', flush=True)

# Clear pyc cache
import os, glob
for d in ['/usr/local/lib/python3.12/dist-packages/numpy']:
    for f in glob.glob(d + '/**/*.pyc', recursive=True):
        try: os.remove(f)
        except: pass
print('[patch] All numpy patches applied', flush=True)

# Install kev
!pip install -q git+https://github.com/jaredpalmer/kev.git 2>&1 | tail -3
print('kev installed', flush=True)

# Import everything
import torch
print(f'torch: {torch.__version__}', flush=True)
print(f'GPU: {torch.cuda.get_device_name(0)}', flush=True)

import kev
import kev.model
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'kev: {kev.__file__}', flush=True)
print('All imports OK!', flush=True)


In [ ]:
import json, os

data_path = '/tmp/wcag_train.jsonl'

if not os.path.exists(data_path):
    kaggle_path = '/kaggle/input/wcag-kev-training/wcag_train.jsonl'
    if os.path.exists(kaggle_path):
        data_path = kaggle_path
    else:
        print('Creating 20 sample records...')
        wcag_data = [{'questions': [{'type': 'WCAG', 'prompt': f'Q{i+1}', 'label': '4.5:1', 'msetup': 'wcag2.1'}]} for i in range(20)]
        with open('/tmp/wcag_train.jsonl', 'w') as f:
            for item in wcag_data:
                f.write(json.dumps(item) + chr(10))
        data_path = '/tmp/wcag_train.jsonl'

with open(data_path) as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f'Loaded {len(records)} records', flush=True)


In [ ]:
print('Loading Qwen2-0.5B...', flush=True)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2-0.5B', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2-0.5B', trust_remote_code=True)
print(f'Model: {model.__class__.__name__}', flush=True)

from kev.data import load_records
train_records = load_records(data_path)
print(f'Training records: {len(train_records)}', flush=True)

print('Training...', flush=True)
dm = kev.model.DecisionModel('Qwen/Qwen2-0.5B', 'cuda', lora=16, head_dim=256)
dm.train()
print('Training complete!', flush=True)


In [ ]:
output_dir = '/tmp/kev-wcag-model'
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Model saved to {output_dir}', flush=True)

from pathlib import Path
for f in sorted(Path(output_dir).glob('**/*')):
    if f.is_file():
        print(f'  {f.name}: {os.path.getsize(f)/1024:.1f} KB', flush=True)

print('DONE!', flush=True)
